# Predicción del promedio semanal del nivel máximo del río Magdalena

## I. Descripción del problema e inspección de los datos
   ### 1. Descripción del problema.
   
   Este notebook es un ejercicio de aprendizaje de machine learning.
   Trabajamos con el nivel maximo del río Magdalena (medición horaria) para practicar preparación de datasets, evaluación de modelos de aprendizaje supervisado y diseño de experimentos.

   El río Magdalena tiene una extensión de 1.540 kilómetros y atraviesa hasta 13 departamentos del territorio colombiano. Su nivel se mide con sensores. *Este dataset fue extraído desde la página oficial de Datos Abiertos Colombia y se puede obtener en el siguiente enlace: [Dataset utilizado](https://www.datos.gov.co/Ambiente-y-Desarrollo-Sostenible/Nivel-M-ximo-del-Rio/vfth-yucv/explore/query/SELECT%20%60fechaobservacion%60%2C%20%60valorobservado%60%0AWHERE%20caseless_one_of%28%60codigoestacion%60%2C%20"0025027020"%29/page/filter).*

   ### 2. Los datos (variables, número de casos, fuente).
   
   Tenemos una **serie de tiempo**: un sensor registró el nivel del agua en
   distintos horarios. Los datos están en `Nivel_Máximo_del_Rio_20261026.csv`
   y cubren desde el 4 de octubre de 2004 hasta el 26 de octubre de 2026, de la estación EL BANCO - AUT (código 25027020).
   


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:

df = pd.read_csv("Nivel_Máximo_del_Rio_20261026.csv")

df.info()
print(f"Filas: {df.shape[0]}  Columnas: {df.shape[1]}")
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 105428 entries, 0 to 105427
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   FechaObservacion  105428 non-null  str  
 1   ValorObservado    105428 non-null  str  
dtypes: str(2)
memory usage: 1.6 MB
Filas: 105428  Columnas: 2


,FechaObservacion,ValorObservado
0,2019 Jun 28 10:08:00 PM,"7,44"
1,2015 Mar 11 04:37:00 PM,"4,83"
2,2018 Nov 13 01:42:00 AM,"8,71"
3,2017 Nov 06 03:56:00 PM,"6,66"
4,2017 Nov 21 01:39:00 PM,"5,8"


### Carga y revisión inicial de los datos

**Qué se hizo:** se cargó el archivo y se creo el dataframe para inspeccionar en detalle el data set.

| Dato | Valor |
|------|-------|
| Registros | 105.428 |
| Columnas | 2 |
| Valores vacíos | 0 |
| Tipo de datos | Texto (string) |

**Observaciones**

Contamos con 2 columnas, la fecha y el nivel máximo registrado, ambas de tipo texto (*string*), y 105.428 registros.

- La fecha tiene el siguiente formato Año, mes, día y hora, separados por espacios y no por guiones, el mes va en texto abreviado en inglés (Jun, Nov, Mar) y la hora usa formato de 12 horas con AM/PM; el valor registrado tiene como separador de decimal "," por esto se reconocen como tipo de dato string.
- El dataset no se encuentra ordenado.

> **Decisión:** Se ordenaran las filas del data set y se convertirán el tipo de dato para la fecha en datetime y el valor registrado en un tipo de dato float.

In [7]:
df["ValorObservado"] = df["ValorObservado"].str.replace(",", ".")
df["ValorObservado"] = pd.to_numeric(df["ValorObservado"], errors="coerce")

print(f"Datos valor observado con errores al convertir: {df["ValorObservado"].isna().sum()}")

df["FechaObservacion"] = pd.to_datetime(df["FechaObservacion"], format="%Y %b %d %I:%M:%S %p", errors="coerce")

print(f"Datos fecha observada con errores al convertir: {df["FechaObservacion"].isna().sum()}")

df.info()

Datos valor observado con errores al convertir: 0
Datos fecha observada con errores al convertir: 0
<class 'pandas.DataFrame'>
RangeIndex: 105428 entries, 0 to 105427
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   FechaObservacion  105428 non-null  datetime64[us]
 1   ValorObservado    105428 non-null  float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 1.6 MB


Con esto podemos ver que ahora tenemos los tipos de datos correcto para poder trabajar sobre ellos y que no hubo ningún error de conversion.

### Inspección: calidad de los datos

In [8]:
df_filtrado = df.sort_values("FechaObservacion").reset_index(drop=True)

filas_identicas = df_filtrado.duplicated().sum()

fechas_repetidas = df_filtrado.duplicated(subset=["FechaObservacion"]).sum()

conteo = df_filtrado["FechaObservacion"].value_counts()
conteo = conteo[conteo > 1]

print("Filas idénticas:", filas_identicas)
print("Filas con fecha repetida (sin la primera):", fechas_repetidas)
print("Fechas distintas que se repiten:", len(conteo))

df_filtrado[df_filtrado.duplicated(subset=["FechaObservacion"], keep=False)]


Filas idénticas: 6125
Filas con fecha repetida (sin la primera): 6125
Fechas distintas que se repiten: 5523


,FechaObservacion,ValorObservado
49248,2019-08-28 00:02:00,6.39
49249,2019-08-28 00:02:00,6.39
49250,2019-08-28 00:02:00,6.39
49251,2019-08-28 01:01:00,6.39
49252,2019-08-28 01:01:00,6.39
...,...,...
103717,2026-07-24 23:51:00,4.91
103718,2026-07-24 23:51:00,4.91
103719,2026-07-24 23:51:00,4.91
103720,2026-07-24 23:51:00,4.91


Hay 6125 filas que son copias exactas de otra fila (misma fecha y mismo valor), y están repartidas en 5523 fechas distintas

No se encontró ninguna misma fecha con valores diferentes.

Si se eliminaran las copias, quedarían 105.428 − 6125 = 99.303 filas.

In [9]:
df_unico = df_filtrado.drop_duplicates().reset_index(drop=True)

print("Antes:", df_filtrado.shape[0])
print("Después:", df_unico.shape[0])
print("Eliminadas:", df_filtrado.shape[0] - df_unico.shape[0])
print("Fechas que aún se repiten:", df_unico.duplicated(subset=["FechaObservacion"]).sum())

Antes: 105428
Después: 99303
Eliminadas: 6125
Fechas que aún se repiten: 0


Con nuestro dataset libre de datos erroneos y/o repetidos continuamos analizando la frecuencia en la que son tomados los datos.

In [11]:
diferencias = df_unico["FechaObservacion"].diff()
print("Frecuencia en que se toma\n", diferencias.value_counts().head(10))
print("Intervalo más pequeño:", diferencias.min())
print("Intervalo más grande:", diferencias.max())

df_unico.head(50)


Frecuencia en que se toma
 FechaObservacion
0 days 01:00:00    12315
0 days 00:02:00     6896
0 days 00:04:00     2227
0 days 00:59:00     1782
0 days 01:01:00     1707
0 days 00:58:00     1479
0 days 00:57:00     1361
0 days 01:02:00     1359
0 days 00:56:00     1335
0 days 01:03:00     1258
Name: count, dtype: int64
Intervalo más pequeño: 0 days 00:01:00
Intervalo más grande: 2793 days 23:29:00


,FechaObservacion,ValorObservado
0,2004-10-04 22:54:03,0.44
1,2012-05-29 22:23:03,29.24
2,2015-01-01 00:01:00,5.72
3,2015-01-01 01:11:00,5.72
4,2015-01-01 02:52:00,5.73
5,2015-01-01 03:12:00,5.72
6,2015-01-01 04:29:00,5.73
7,2015-01-01 05:50:00,5.73
8,2015-01-01 06:45:00,5.73
9,2015-01-01 07:12:00,5.73


Se identifica que hay un gran numero de registros que son tomados con una frecuencia de una hora (12315)

   4. Inspección: gráficas y análisis

## II. Diseño de experimentos y resultados
   5. Preparación de los datos
   6. Modelos y ajuste de hiperparámetros
   7. Resultados de cada modelo

## III. Comparación y conclusiones
   8. Comparación entre los modelos
   9. Conclusiones, trabajo futuro y ética

## Bibliografía